# PAS: treino no Kaggle

Treina o PVTv2 + EMCAD no protocolo padrão (Kvasir-SEG + CVC-ClinicDB no treino e 5 conjuntos de teste) e avalia o resultado.

**Antes de rodar:** em *Settings*, coloque *Accelerator = GPU T4 x2* (ou uma T4) e ative *Internet = On*.
Se o repositório for privado, crie um *Secret* `GITHUB_TOKEN` (em *Add-ons → Secrets*) com um token de leitura (fine-grained, *Contents: Read-only*, resource owner `warrior-graph`). O token nunca aparece nas mensagens de erro.

Tempo estimado em uma T4: B0 por 100 épocas leva cerca de 1 hora; B2 leva de 2 a 3 horas.

In [ ]:
CONFIG = "configs/emcad_b0.yaml"   # ou configs/emcad_b2.yaml
EPOCHS = 100
BATCH_SIZE = 16                    # T4 (16 GB): 16 para B0 e B2
BRANCH = "claude/gifted-albattani-ikhwce"  # mude para "main" depois do merge
REPO = "warrior-graph/PAS-PolypAutoSegmentation"

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Sem GPU: Settings -> Accelerator -> GPU T4 x2 (exige telefone verificado)"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import os, shutil, subprocess, urllib.error, urllib.request

DEST = "/kaggle/working/PAS"
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass


def git(*args, cwd=None):
    """Run git, never echoing the token in errors."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        msg = (r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr
        hints = {
            "Could not resolve host": "Sem Internet: ative Settings -> Internet: On.",
            "could not read Username": "Sem token: anexe o secret GITHUB_TOKEN a este notebook (Add-ons -> Secrets) ou torne o repo público.",
            "Repository not found": "Token sem acesso ao repo (resource owner, Contents: Read, aprovação da org).",
            "Authentication failed": "Token inválido/expirado ou sem acesso ao repo.",
            "not found in upstream": "Branch inexistente: confira BRANCH.",
            "couldn't find remote ref": "Branch inexistente: confira BRANCH.",
        }
        hint = next((h for k, h in hints.items() if k in msg), "")
        raise RuntimeError(f"git {args[0]} falhou:\n{msg}\n{hint}") from None
    return r.stdout


try:
    urllib.request.urlopen("https://github.com", timeout=10)
except urllib.error.HTTPError:
    pass  # respondeu: há Internet
except Exception:
    raise RuntimeError("Sem acesso à Internet: ative Settings -> Internet: On (exige telefone verificado).") from None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
try:
    if os.path.isdir(os.path.join(DEST, ".git")):
        # Clone anterior: atualiza para a última versão do branch
        git("remote", "set-url", "origin", url, cwd=DEST)
        git("fetch", "--depth", "1", "origin", BRANCH, cwd=DEST)
        git("reset", "--hard", "FETCH_HEAD", cwd=DEST)
    else:
        shutil.rmtree(DEST, ignore_errors=True)  # resto de um clone que falhou
        git("clone", "-b", BRANCH, "--depth", "1", url, DEST)
finally:
    # /kaggle/working vai para o Output: nunca deixar o token no .git/config
    if os.path.isdir(os.path.join(DEST, ".git")):
        subprocess.run(["git", "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], cwd=DEST)
print(git("log", "--oneline", "-1", cwd=DEST))
os.chdir(DEST)


def run(cmd):
    """Run a shell command streaming its output; fail the cell (and a Save & Run All) on error."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"comando falhou ({p.returncode}): {cmd}")

In [ ]:
run("pip install -q -U timm 'albumentations>=2.0' gdown")

In [ ]:
# Dados fora de /kaggle/working para não irem ao Output (~700 MB)
DATA = "/tmp/pas_data"
run(f"python scripts/download_data.py --root {DATA}")

In [ ]:
EXP = os.path.splitext(os.path.basename(CONFIG))[0]
run(f"python train.py --config {CONFIG} --set data.root={DATA} train.epochs={EPOCHS} "
    f"train.batch_size={BATCH_SIZE} train.grad_accum=1 data.num_workers=4")

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"runs/{EXP}/test_results.md").read()))

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f"runs/{EXP}/log.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
log.plot(x="epoch", y="train_loss", ax=ax[0], title="train loss")
log.plot(x="epoch", y="val_mDice", ax=ax[1], title="val Dice (10% do TrainDataset)")
plt.show()

In [ ]:
# Avaliação com TTA (flip horizontal)
run(f"python eval.py --checkpoint runs/{EXP}/best.pth --tta")
display(Markdown(open(f"runs/{EXP}/eval_tta.md").read()))

In [ ]:
# Visualização qualitativa no ETIS (o conjunto mais difícil)
import glob, cv2
run(f"python predict.py --checkpoint runs/{EXP}/best.pth --input {DATA}/TestDataset/ETIS-LaribPolypDB/images --output /tmp/etis_preds > /dev/null")
files = sorted(glob.glob("/tmp/etis_preds/*_overlay.png"))[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, f in zip(axes.ravel(), files):
    ax.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); ax.set_title(os.path.basename(f)); ax.axis("off")
plt.show()

In [ ]:
# Empacota checkpoint + resultados para download (aba Output)
!cd runs && zip -qr /kaggle/working/{EXP}_run.zip {EXP}
!ls -lh /kaggle/working/*.zip